<a href="https://colab.research.google.com/github/PavithraGanapathi/Internship-repo/blob/main/Fine_tunig_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/Labeled Dataset/dataset"




 AI_01	 AI_07	   CD1_075   CD1_093   CD1_107	 CD1_124
 AI_02	 CD1_065   CD1_078   CD1_100   CD1_112	 CD1_91
 AI_03	 CD1_066   CD1_079   CD1_101   CD1_118	'DPSD1 003'
 AI_04	 CD1_067   CD1_082   CD1_104   CD1_122	'DPSD1 007'
 AI_06	 CD1_068   CD1_087   CD1_106   CD1_123


In [3]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments


In [4]:
import os
import pandas as pd

csv_dir = "/content/drive/MyDrive/Labeled Dataset/csv dataset/"
drive_base = "/content/drive/MyDrive/Labeled Dataset/dataset/"
output_path = "/content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv"

dfs = []

for file in os.listdir(csv_dir):
    if not file.endswith(".csv"):
        continue

    path = os.path.join(csv_dir, file)
    df = pd.read_csv(path)

    # Detect text column
    if "text" in df.columns:
        text_col = "text"
    elif "corrected_text" in df.columns:
        text_col = "corrected_text"
    else:
        continue

    # Detect image column
    if "full_image_path" in df.columns:
        img_col = "full_image_path"
    elif "image_path" in df.columns:
        img_col = "image_path"
    else:
        continue

    # Keep only relevant columns
    df = df[["line_id", text_col, img_col]].rename(columns={text_col: "text", img_col: "image_path"})

    # Remove empty/placeholder text
    df = df[~df["text"].isin(["", "0 0", "0 0 "])]

    # Function to safely fix path
    def safe_fix_path(row):
        p = str(row["image_path"]).replace("\\", "/")  # convert backslashes to slashes
        p = p.replace("C:/Users/pavit/Desktop", drive_base)
        p = p.replace("/home/appuser/Desktop", drive_base)

        # Extract folder name from line_id
        folder = row["line_id"].split(".pdf")[0] if pd.notna(row["line_id"]) else "UNKNOWN"
        fname = os.path.basename(p) if p else "UNKNOWN_FILE"
        new_path = os.path.join(drive_base, folder, fname)
        return new_path

    # Apply row-wise
    df["image_path"] = df.apply(safe_fix_path, axis=1)

    dfs.append(df)

# Merge all CSVs
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.drop_duplicates(subset=["line_id"], inplace=True)
combined_df.reset_index(drop=True, inplace=True)

# Check how many images actually exist
existing = combined_df["image_path"].apply(os.path.exists)
print(f"Total rows: {len(combined_df)}, Existing images: {existing.sum()}, Missing: {len(combined_df) - existing.sum()}")

# Save cleaned dataset
combined_df.to_csv(output_path, index=False)
print(f"✅ Clean dataset saved at: {output_path}")


Total rows: 404, Existing images: 355, Missing: 49
✅ Clean dataset saved at: /content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv


In [5]:
missing = combined_df[~combined_df["image_path"].apply(os.path.exists)]
for p in missing["image_path"].head(10):
    print(repr(p))


'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page005_line001.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page007_line003.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line002.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line003.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line004.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line006.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line007.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line008.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page013_line001.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page013_line002.png'


**1️⃣ Install & Import**

In [6]:
!pip install transformers datasets evaluate torch torchvision pillow --quiet

import os
import torch
import pandas as pd
from PIL import Image
from datasets import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate


**2️⃣ Load the cleaned dataset**

In [7]:
clean_csv = "/content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv"
df = pd.read_csv(clean_csv)

# Keep only rows where images exist
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

# Split train/validation (approx 90-10 split)
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

print(f"Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)


Train samples: 320, Validation samples: 35


**3️⃣ Load Processor & Model**

In [15]:
!pip uninstall -y wandb

In [16]:
from transformers import Trainer, TrainingArguments
import os

# Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"



In [8]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Set decoder parameters
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Optional: Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['enco

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

**4️⃣ Preprocessing Function**

In [9]:
max_target_length = 128

from transformers import TrOCRProcessor
from PIL import Image
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

def preprocess(example):
    # Load image
    image = Image.open(example["image_path"]).convert("RGB")

    # Encode image only
    pixel_values = processor.feature_extractor(images=image, return_tensors="pt").pixel_values[0]

    # Encode target text separately
    text = str(example["text"]) if example["text"] is not None else ""
    labels = processor.tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]

    # Replace pad token with -100 (ignore padding in loss)
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }





# Apply preprocessing
train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

train_ds.set_format(type="torch", columns=["pixel_values", "labels"])
val_ds.set_format(type="torch", columns=["pixel_values", "labels"])


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

**5️⃣ Data Collecctor**

In [10]:
!pip install jiwer



In [11]:
import torch
from torch.nn.utils.rnn import pad_sequence

def data_collator(batch):
    pixel_values = [example["pixel_values"] for example in batch]
    labels = [example["labels"] for example in batch]

    # Stack pixel_values manually
    pixel_values = torch.stack(pixel_values)

    # Pad labels to max length in batch
    labels = pad_sequence([torch.tensor(l) for l in labels], batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {"pixel_values": pixel_values, "labels": labels}



**2️⃣ Metric (CER)**

In [12]:
import evaluate

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    # Decode labels
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


**3️⃣ Training Arguments**

In [13]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_strategy="no",  # disable logging
    save_total_limit=2,
    num_train_epochs=3,
)



**4️⃣ Trainer**

In [14]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=processor,  # works in v4.5.7
    compute_metrics=compute_metrics
)


/tmp/ipython-input-4279257113.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


**5️⃣ Start Training**

In [17]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.
/tmp/ipython-input-2067349078.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = pad_sequence([torch.tensor(l) for l in labels], batch_first=True, padding_value=processor.tokenizer.pad_token_id)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=240, training_loss=4.5182032267252605, metrics={'train_runtime': 251.2236, 'train_samples_per_second': 3.821, 'train_steps_per_second': 0.955, 'total_flos': 7.183537833561293e+17, 'train_loss': 4.5182032267252605, 'epoch': 3.0})

In [24]:
from transformers import TrOCRProcessor

# Folder path
save_dir = "/content/drive/MyDrive/trocr_model_final"

# Save model
model.save_pretrained(save_dir)

# Save processor (this saves both tokenizer + feature extractor together)
processor.save_pretrained(save_dir)

print("✅ Model and processor saved correctly at:", save_dir)


✅ Model and processor saved correctly at: /content/drive/MyDrive/trocr_model_final


**Evaluate the Model**

In [25]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Correct path to where you saved the model in Drive
model_path = "/content/drive/MyDrive/trocr_model_final"

# Reload processor (tokenizer + feature extractor)
processor_reloaded = TrOCRProcessor.from_pretrained(model_path)

# Reload model
model_reloaded = VisionEncoderDecoderModel.from_pretrained(model_path)

print("✅ Processor and model loaded successfully from Google Drive!")



✅ Processor and model loaded successfully from Google Drive!


**Testing**

In [31]:
import os

path = "/content/drive/MyDrive/Labeled Dataset/dataset/DPSD1 007/DPSD1 007.pdf_page004_line019.png"

print(os.path.exists(path))


True


In [34]:
from PIL import Image
import torch

# Use your actual image path
image_path = "/content/drive/MyDrive/Labeled Dataset/dataset/AI_02/AI_02.pdf_page002_line005.png"

# Load and preprocess
image = Image.open(image_path).convert("RGB")
pixel_values = processor_reloaded(images=image, return_tensors="pt").pixel_values

# Generate prediction
generated_ids = model_reloaded.generate(pixel_values)
predicted_text = processor_reloaded.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("🧠 Predicted Text:", predicted_text)


🧠 Predicted Text: 5advant AnalysisCCC.


**Accuracy**

In [1]:
!pip install evaluate
!pip install datasets --upgrade


In [3]:
import torch
import evaluate
from tqdm import tqdm

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")
print("Metrics loaded successfully!")



Metrics loaded successfully!


In [5]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Reload processor and model from Drive
processor_reloaded = TrOCRProcessor.from_pretrained("/content/drive/MyDrive/trocr_model_final")
model_reloaded = VisionEncoderDecoderModel.from_pretrained("/content/drive/MyDrive/trocr_model_final")

print("✅ Model and processor reloaded successfully!")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Model and processor reloaded successfully!


In [7]:
!pip install evaluate


In [2]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

# 🔁 Reload your fine-tuned model and processor from Drive
model_reloaded = VisionEncoderDecoderModel.from_pretrained("/content/drive/MyDrive/trocr_model_final")
processor_reloaded = TrOCRProcessor.from_pretrained("/content/drive/MyDrive/trocr_model_final")

model_reloaded.eval()  # Set model to evaluation mode


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [14]:
# 1️⃣ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2️⃣ Imports
import torch
import os
import pandas as pd
from tqdm import tqdm
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from evaluate import load

# 3️⃣ Load model & processor
model_path = "/content/drive/MyDrive/trocr_model_final"
processor_reloaded = TrOCRProcessor.from_pretrained(model_path)
model_reloaded = VisionEncoderDecoderModel.from_pretrained(model_path)
model_reloaded.eval()

# 4️⃣ Load CSV validation data
csv_path = "/content/drive/MyDrive/Labeled Dataset/val.csv"
val_df = pd.read_csv(csv_path)

# 🧩 Base path for images
base_path = "/content/drive/MyDrive/Labeled Dataset/dataset/"

# 🧹 Fix image paths properly
def fix_path(x):
    x = str(x).strip()

    # If path doesn't start with "/content/drive", add base path
    if not x.startswith("/content/drive"):
        x = os.path.join("/content/drive/MyDrive/Labeled Dataset", x)

    # Replace wrong "MyDrive/dataset" with correct "MyDrive/Labeled Dataset/dataset"
    x = x.replace("/content/drive/MyDrive/dataset/", "/content/drive/MyDrive/Labeled Dataset/dataset/")

    # Remove accidental double "dataset/dataset"
    x = x.replace("dataset/dataset", "dataset")

    return x

val_df["image_path"] = val_df["image_path"].apply(fix_path)

# ✅ Check how many files exist
val_df["exists"] = val_df["image_path"].apply(os.path.exists)
print(val_df["exists"].value_counts())

# Only keep existing images
val_df = val_df[val_df["exists"]]
print(f"✅ Using {len(val_df)} valid images for evaluation.")


# 5️⃣ Load accuracy metric
accuracy_metric = load("accuracy")

# 6️⃣ Evaluate on CSV data
pred_texts = []
true_texts = []

for idx, row in tqdm(val_df.iterrows(), total=len(val_df)):
    image_path = row["image_path"]
    true_text = str(row["text"]).strip()

    image = Image.open(image_path).convert("RGB")
    pixel_values = processor_reloaded(images=image, return_tensors="pt").pixel_values

    with torch.no_grad():
        generated_ids = model_reloaded.generate(pixel_values)
        pred_text = processor_reloaded.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    pred_texts.append(pred_text)
    true_texts.append(true_text)


from evaluate import load

# Load text-based metrics
wer_metric = load("wer")
cer_metric = load("cer")

# Compute metrics
wer = wer_metric.compute(references=true_texts, predictions=pred_texts)
cer = cer_metric.compute(references=true_texts, predictions=pred_texts)

print("✅ Evaluation Results:")
print(f"Word Error Rate (WER): {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")

# Optional: Approximate text accuracy
text_accuracy = (1 - cer) * 100
print(f"Approximate Text Accuracy: {text_accuracy:.2f}%")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
exists
True     35
False     6
Name: count, dtype: int64
✅ Using 35 valid images for evaluation.


100%|██████████| 35/35 [02:41<00:00,  4.62s/it]


✅ Evaluation Results:
Word Error Rate (WER): 1.1214
Character Error Rate (CER): 0.7215
Approximate Text Accuracy: 27.85%
